In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

In [2]:
# ---------------------------------------------------------
# 1. Datos de entrada (Stocks, Bonds, Real Estate)
# ---------------------------------------------------------
asset_names = ['Stocks', 'Bonds', 'Real E.']

# Retornos esperados (E) y Desviación Estándar (sigma)
E = np.array([0.065, 0.040, 0.050])
sigma = np.array([0.16, 0.07, 0.12])

# Matriz de covarianza (\Omega) calculada/dada en el problema
Omega = np.array([
    [0.02560, 0.00392, 0.00384],
    [0.00392, 0.00490, 0.00168],
    [0.00384, 0.00168, 0.01440]
])

# Vector de unos
ones = np.ones(len(E))

In [36]:
# ---------------------------------------------------------
# 2. Portafolio Personalizado (wS = 0.4, wB = 0.1, wR = 0.5)
# ---------------------------------------------------------
w_custom = np.array([0.9, 0.05, 0.05])

# Retorno y volatilidad del portafolio personalizado
E_custom = float(w_custom.T @ E)
sigma_custom = float(np.sqrt(w_custom.T @ Omega @ w_custom))
print("-" * 55)
print(f"Portafolio Evaluado (0.4, 0.1, 0.5):")
print(f"  - Retorno Esperado : {E_custom:.2%}")
print(f"  - Volatilidad (σ)  : {sigma_custom:.2%}")
print("=" * 55)

-------------------------------------------------------
Portafolio Evaluado (0.4, 0.1, 0.5):
  - Retorno Esperado : 6.30%
  - Volatilidad (σ)  : 14.66%


In [37]:
# Inversa de la matriz de covarianza
Omega_inv = np.linalg.inv(Omega)

# Componentes de la matriz A
a11 = float(E.T @ Omega_inv @ E)
a12 = float(ones.T @ Omega_inv @ E)  # igual a E.T @ Omega_inv @ ones
a22 = float(ones.T @ Omega_inv @ ones)

# Matriz A
A = np.array([
    [a11, a12],
    [a12, a22]
])

# Determinante de A (\Delta = a11 * a22 - a12^2)
delta = np.linalg.det(A)

In [38]:
# ---------------------------------------------------------
# Impresión de Resultados
# ---------------------------------------------------------
print("=" * 45)
print("              RESULTADOS DE LA MATRIZ A           ")
print("=" * 45)
print(f"a11 (E' Ω^-1 E) : {a11:.6f}")
print(f"a12 (1' Ω^-1 E) : {a12:.6f}")
print(f"a22 (1' Ω^-1 1) : {a22:.6f}\n")
print("Matriz A:")
print(pd.DataFrame(A, columns=['E', '1'], index=['E', '1']))
print(f"\nDeterminante Delta (Δ): {delta:.6f}")
print("=" * 45)

              RESULTADOS DE LA MATRIZ A           
a11 (E' Ω^-1 E) : 0.453552
a12 (1' Ω^-1 E) : 9.985895
a22 (1' Ω^-1 1) : 235.620444

Matriz A:
          E           1
E  0.453552    9.985895
1  9.985895  235.620444

Determinante Delta (Δ): 7.147933


In [39]:
# ---------------------------------------------------------
# 3. Construcción de la Frontera Eficiente
# ---------------------------------------------------------
# Rango de retornos esperados (\mu / E_p) para graficar la curva
mu_vals = np.linspace(0.01, 0.10, 500)

# Fórmula exacta para la varianza en función del retorno mu:
# \sigma_P^2 = (a22 / \Delta) * (\mu - a12 / a22)^2 + (1 / a22)
sigma2_p = (a22 / delta) * (mu_vals - (a12 / a22))**2 + (1 / a22)
sigma_p = np.sqrt(sigma2_p)

# Separar la frontera eficiente (parte superior) de la no eficiente
mu_min_var = a12 / a22
sigma_min_var = np.sqrt(1 / a22)

In [49]:
# ---------------------------------------------------------
# 4. Visualización con Plotly
# ---------------------------------------------------------
fig = go.Figure()

# Curva completa de mínima varianza (hipérbola)
fig.add_trace(go.Scatter(
    x=sigma_p,
    y=mu_vals,
    mode='lines',
    name='Frontera de Mínima Varianza',
    line=dict(color='gray', dash='dash', width=2),
    hovertemplate='Volatilidad (σ): %{x:.2%}<br>Retorno (E): %{y:.2%}'
))

# Parte eficiente (Retorno >= mu del portafolio de mínima varianza)
mask_efficient = mu_vals >= mu_min_var
fig.add_trace(go.Scatter(
    x=sigma_p[mask_efficient],
    y=mu_vals[mask_efficient],
    mode='lines',
    name='Frontera Eficiente',
    line=dict(color='#00CC96', width=4),
    hovertemplate='Volatilidad (σ): %{x:.2%}<br>Retorno (E): %{y:.2%}'
))

# Portafolio de Mínima Varianza Global (GMVP)
fig.add_trace(go.Scatter(
    x=[sigma_min_var],
    y=[mu_min_var],
    mode='markers',
    name='Mínima Varianza Global (GMVP)',
    marker=dict(size=12, color='gold', symbol='star'),
    hovertemplate='<b>GMVP</b><br>Volatilidad: %{x:.2%}<br>Retorno: %{y:.2%}'
))

# Activos individuales
fig.add_trace(go.Scatter(
    x=sigma,
    y=E,
    mode='markers+text',
    name='Activos',
    text=asset_names,
    textposition="top center",
    marker=dict(size=10, color='crimson'),
    hovertemplate='<b>%{text}</b><br>Volatilidad: %{x:.2%}<br>Retorno: %{y:.2%}'
))

# Portafolio Específico
fig.add_trace(go.Scatter(
    x=[sigma_custom],
    y=[E_custom],
    mode='markers+text',
    name='Portafolio [0.4, 0.1, 0.5]',
    text=['Portafolio A'],
    textposition="bottom right",
    marker=dict(size=12, color='white', symbol='star', line=dict(width=0, color='white')),
    hovertemplate='<b>Portafolio [0.4, 0.1, 0.5]</b><br>Volatilidad: %{x:.2%}<br>Retorno: %{y:.2%}'
))

# Configuración del diseño del gráfico
fig.update_layout(
    title='<b>Frontera Eficiente de Markowitz</b>',
    xaxis_title='Riesgo / Volatilidad (σ)',
    yaxis_title='Retorno Esperado (E)',
    xaxis=dict(tickformat='.1%'),
    yaxis=dict(tickformat='.1%'),
    template='plotly_dark',
    hovermode='closest',
    width=850,
    height=600
)

# Mostrar el gráfico
fig.show()